# Example: Image Serving with Azure Blob Knowledge Source

This notebook creates an Azure Blob-backed agentic retrieval pipeline on Foundry IQ (Azure AI Search) with **image serving** enabled.
Image serving delivers extracted image content (as hosted URLs) directly to the downstream model during retrieval,
so a multimodal chat model can reason over both text and images in one request.

**Pre-requisites:**
1. Follow the steps in the [image serving documentation](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-image-serving) for permissions requirements prior to running this sample.
2. This sample uses token authentication, so it needs a registered application to generate the tokens. You can also modify for Foundry IQ keys.

**Flow:**
1. Create an `azureBlob` knowledge source with `content_extraction_mode="standard"` — images are extracted via
   Azure AI Services, persisted in a Blob asset-store container, and verbalized by a chat model at ingestion time.
2. Poll until the knowledge source completes its first synchronization.
3. Create a knowledge base with `enable_image_serving=True` on the knowledge-source reference.
4. Retrieve — pass `enable_image_serving=True` on `AzureBlobKnowledgeSourceParams` so the service delivers
   image URLs alongside text in the response messages.
5. Inspect `ImageServingStatistics` from the activity records.
6. Clean up.

**SDK:** `azure-search-documents==12.1.0b1` (REST API `2026-05-01-preview`)

Save `sample.env` as `.env`, fill in the values, and create a virtual environment from `requirements.txt`
before running this notebook.

## Load connections

Before you run this cell, save `sample.env` as `.env` and fill in the values.
You should also create a virtual environment with `requirements.txt` as the dependency list
and select it as the notebook kernel.

In [ ]:
import os
import time

from azure.identity import ClientSecretCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint                    = os.environ["AZURE_SEARCH_ENDPOINT"]
client_id                   = os.environ["AZURE_CLIENT_ID"]
client_secret               = os.environ["AZURE_CLIENT_SECRET"]
tenant_id                   = os.environ["AZURE_TENANT_ID"]
ks_name                     = os.getenv("AZURE_SEARCH_KS_NAME", "image-serving-ks")
kb_name                     = os.getenv("AZURE_SEARCH_KB_NAME", "image-serving-kb")
blob_connection_string      = os.environ["AZURE_BLOB_CONNECTION_STRING"]
source_container            = os.getenv("AZURE_BLOB_SOURCE_CONTAINER", "source-documents")
asset_connection_string     = os.getenv("AZURE_BLOB_ASSET_CONNECTION_STRING", blob_connection_string)
asset_container             = os.getenv("AZURE_BLOB_ASSET_CONTAINER", "image-assets")
ai_services_endpoint        = os.environ["AZURE_AI_SERVICES_ENDPOINT"]
ai_services_api_key         = os.getenv("AZURE_AI_SERVICES_API_KEY")
openai_endpoint             = os.environ["AZURE_OPENAI_ENDPOINT"]
openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
openai_embedding_model      = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
chat_endpoint               = os.environ["AZURE_OPENAI_CHAT_ENDPOINT"]
chat_api_key                = os.environ["AZURE_OPENAI_CHAT_API_KEY"]
chat_deployment             = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
chat_model                  = os.getenv("AZURE_OPENAI_CHAT_MODEL", "gpt-4o")

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
)
index_client        = SearchIndexClient(endpoint=endpoint, credential=credential)
kb_retrieval_client = KnowledgeBaseRetrievalClient(endpoint=endpoint, credential=credential)

print(f"Endpoint:         {endpoint}")
print(f"Knowledge source: {ks_name}")
print(f"Knowledge base:   {kb_name}")

## Create an Azure Blob knowledge source with image serving

The knowledge source triggers an asynchronous ingestion pipeline that:
1. Crawls the source Blob container.
2. Extracts text and images from each document using Azure AI Services (OCR / Computer Vision).
3. Stores extracted images in the **asset-store** Blob container for later retrieval.
4. Optionally verbalizes images (generates text descriptions) using the chat model as a fallback
   for models that do not support multimodal input.
5. Embeds document text using the Azure OpenAI embedding model.

Key `KnowledgeSourceIngestionParameters` fields used here:
- `content_extraction_mode="standard"` — full text + image extraction (required for image serving).
- `embedding_model` — vectorizes document text chunks.
- `chat_completion_model` — verbalizes images and is re-used for answer synthesis at retrieval time.
- `disable_image_verbalization=False` — keeps verbalization enabled as a fallback.
- `ai_services` — Azure AI Services used for OCR during ingestion.
- `asset_store` — target Blob container where extracted images are persisted.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSource,
    AzureBlobKnowledgeSourceParameters,
    KnowledgeBaseAzureOpenAIModel,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceIngestionParameters,
    KnowledgeSourceAzureOpenAIVectorizer,
    AIServices,
    AssetStore,
)

ks = AzureBlobKnowledgeSource(
    name=ks_name,
    azure_blob_parameters=AzureBlobKnowledgeSourceParameters(
        connection_string=blob_connection_string,
        container_name=source_container,
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            content_extraction_mode="standard",
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                resource_uri=openai_endpoint,
                deployment_id=openai_embedding_deployment,
                model_name=openai_embedding_model,
            ),
            chat_completion_model=KnowledgeBaseAzureOpenAIModel(
                resource_uri=chat_endpoint,
                api_key=chat_api_key,
                deployment_id=chat_deployment,
                model_name=chat_model,
            ),
            disable_image_verbalization=False,
            ai_services=AIServices(
                uri=ai_services_endpoint,
                api_key=ai_services_api_key,
            ),
            asset_store=AssetStore(
                connection_string=asset_connection_string,
                container_name=asset_container,
            ),
        ),
    ),
)

result = index_client.create_or_update_knowledge_source(knowledge_source=ks)
print(f"Knowledge source '{result.name}' created.")
if getattr(result, "created_resources", None):
    for r in result.created_resources:
        print(f"  Created resource: {r}")

## Poll knowledge source sync status

The `azureBlob` ingestion pipeline is asynchronous. The cell below polls every 30 seconds until the
knowledge source reports a terminal status (`success` or `error`). Initial sync for a large container
can take several minutes; image extraction adds extra time compared to text-only pipelines.

In [ ]:
POLL_INTERVAL_SECS = 30
POLL_TIMEOUT_SECS  = 1800  # 30 minutes

start       = time.time()
sync_status = None

while True:
    ks_status   = index_client.get_knowledge_source(ks_name)
    sync_status = (
        getattr(ks_status, "last_sync_status", None)
        or getattr(ks_status, "status", None)
    )
    elapsed = int(time.time() - start)
    print(f"[{elapsed:4d}s] sync_status={sync_status}")

    if sync_status in ("success", "error", "transientFailure"):
        break
    if time.time() - start > POLL_TIMEOUT_SECS:
        print("Timed out waiting for sync. Proceeding anyway.")
        break
    time.sleep(POLL_INTERVAL_SECS)

print(f"Final sync status: {sync_status}")

## Create a knowledge base with image serving enabled

Setting `enable_image_serving=True` on `KnowledgeSourceReference` instructs the service to deliver
extracted images as hosted URLs in retrieval responses. The `gpt-4o` chat model configured on the
knowledge base will then receive both text and image URLs when synthesizing its answer.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalMinimalReasoningEffort,
)

kb = KnowledgeBase(
    name=kb_name,
    knowledge_sources=[
        KnowledgeSourceReference(
            name=ks_name,
            enable_image_serving=True,
        )
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    models=[
        KnowledgeBaseAzureOpenAIModel(
            resource_uri=chat_endpoint,
            api_key=chat_api_key,
            deployment_id=chat_deployment,
            model_name=chat_model,
        )
    ],
)

index_client.create_or_update_knowledge_base(knowledge_base=kb)
print(f"Knowledge base '{kb_name}' created with image serving enabled.")

## Retrieve with image serving

Pass `enable_image_serving=True` on `AzureBlobKnowledgeSourceParams` to request image content in the
retrieval response. When the service matches documents that contain images, it injects image URLs into
the response messages alongside the text answer.

`include_activity=True` attaches per-step diagnostics to the response, including `ImageServingStatistics`
on each Azure Blob activity record.

In [ ]:
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    AzureBlobKnowledgeSourceParams,
)

question = "What do the charts and diagrams in these documents show?"

request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        AzureBlobKnowledgeSourceParams(
            knowledge_source_name=ks_name,
            include_references=True,
            include_reference_source_data=True,
            enable_image_serving=True,
        )
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    include_activity=True,
)

response = kb_retrieval_client.retrieve(
    knowledge_base_name=kb_name,
    body=request,
)

# Display response messages — may contain both text and image content
print("=== Response messages ===")
for msg in (response.response or []):
    role = getattr(msg, "role", "unknown")
    for item in (msg.content or []):
        text = getattr(item, "text", None)
        if text:
            print(f"[{role}] {text}")
        image = getattr(item, "image", None)
        if image:
            url = getattr(image, "url", None)
            if url:
                print(f"[{role}] <image: {url}>")

## Inspect image serving statistics

Each `KnowledgeBaseAzureBlobActivityRecord` in the response carries an `image_serving` field of type
`ImageServingStatistics` when image serving is active. The statistics show how many images were
retrieved from the asset store, how many were passed to the model, and whether verbalization was used
as a fallback.

In [ ]:
from azure.search.documents.knowledgebases.models import (
    ImageServingStatistics,
    KnowledgeBaseAzureBlobActivityRecord,
)

print("=== Image serving statistics ===")
found_stats = False
for record in (response.activity or []):
    image_stats = getattr(record, "image_serving", None)
    if image_stats is not None:
        found_stats = True
        record_type = getattr(record, "type", type(record).__name__)
        print(f"Activity record type  : {record_type}")
        print(f"  images_retrieved    : {getattr(image_stats, 'images_retrieved', 'n/a')}")
        print(f"  images_sent_to_model: {getattr(image_stats, 'images_sent_to_model', 'n/a')}")
        print(f"  verbalization_used  : {getattr(image_stats, 'verbalization_used', 'n/a')}")
        print()

if not found_stats:
    print("No image serving statistics found in this response.")
    print("Ensure the knowledge source has documents with images and image serving is enabled.")

## Clean up

Delete the knowledge base first, then the knowledge source.

### Delete knowledge base

In [ ]:
index_client.delete_knowledge_base(kb_name)
print(f"Knowledge base '{kb_name}' deleted.")

### Delete knowledge source

In [ ]:
index_client.delete_knowledge_source(knowledge_source=ks_name)
print(f"Knowledge source '{ks_name}' deleted.")